# Arabic ID OCR — YOLOv11 Training on Kaggle
**Pipeline:** Roboflow dataset → YOLOv11n training → MLflow tracking → save best.pt

**Before running:**
1. Enable GPU: *Settings → Accelerator → GPU T4 x2*
2. Add your Roboflow API key: *Add-ons → Secrets → Add Secret* → name: `ROBOFLOW_API_KEY`
   - Get your free key at https://app.roboflow.com/settings/api

In [ ]:
# Cell 1 — Install dependencies
!pip install ultralytics mlflow roboflow -q

In [ ]:
# Cell 2 — Verify GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')
print(f'torch: {torch.__version__}')

import ultralytics
print(f'ultralytics: {ultralytics.__version__}')

In [ ]:
# Cell 3 — Download dataset from Roboflow
from kaggle_secrets import UserSecretsClient
from roboflow import Roboflow

# Load API key from Kaggle Secrets (add it via Add-ons → Secrets)
secrets = UserSecretsClient()
api_key = secrets.get_secret('ROBOFLOW_API_KEY')

rf = Roboflow(api_key=api_key)
project = rf.workspace('person-id').project('egyptian-person-id-2-amhk6')
version = project.version(1)
dataset = version.download('yolov8')   # yolov8 format = compatible with yolo11

print(f'Dataset downloaded to: {dataset.location}')

In [ ]:
# Cell 4 — Inspect dataset structure
import os, yaml

data_yaml = os.path.join(dataset.location, 'data.yaml')
with open(data_yaml) as f:
    data_cfg = yaml.safe_load(f)

print(f'Classes ({data_cfg["nc"]}):', data_cfg['names'])
print(f'Train images: {len(os.listdir(os.path.join(dataset.location, "train", "images")))}')
print(f'Valid images: {len(os.listdir(os.path.join(dataset.location, "valid", "images")))}')
print(f'Test  images: {len(os.listdir(os.path.join(dataset.location, "test",  "images")))}')

In [ ]:
# Cell 5 — Training config
EPOCHS      = 10        # smoke test; set to 100 for full training
BATCH_SIZE  = 16
IMG_SIZE    = 640
DEVICE      = 0         # GPU 0
WORKERS     = 2
PROJECT     = '/kaggle/working/runs/train'
RUN_NAME    = 'arabic_id_detector'

print(f'Training {EPOCHS} epochs on GPU {DEVICE}')

In [ ]:
# Cell 6 — Train with MLflow tracking
import mlflow
from ultralytics import YOLO
from pathlib import Path

mlflow.set_tracking_uri('/kaggle/working/mlruns')
mlflow.set_experiment('arabic-ocr-detection')

with mlflow.start_run(tags={
    'project': 'arabic-id-ocr',
    'compute': 'kaggle-t4-gpu',
    'dataset': 'egyptian-person-id-v1'
}) as run:

    print(f'MLflow run ID: {run.info.run_id}')

    mlflow.log_params({
        'model': 'yolo11n',
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'img_size': IMG_SIZE,
        'device': f'GPU {DEVICE}',
    })

    model = YOLO('yolo11n.pt')

    results = model.train(
        data=data_yaml,
        epochs=EPOCHS,
        batch=BATCH_SIZE,
        imgsz=IMG_SIZE,
        device=DEVICE,
        workers=WORKERS,
        lr0=0.01,
        lrf=0.01,
        momentum=0.937,
        weight_decay=0.0005,
        warmup_epochs=3,
        patience=50,
        save_period=5,
        flipud=0.5,
        fliplr=0.0,      # ID cards are not horizontally symmetric
        mosaic=1.0,
        project=PROJECT,
        name=RUN_NAME,
        exist_ok=True,
    )

    # Log final metrics to MLflow
    if hasattr(results, 'results_dict'):
        m = results.results_dict
        mlflow.log_metrics({
            'mAP50':     m.get('metrics/mAP50(B)', 0),
            'mAP50_95':  m.get('metrics/mAP50-95(B)', 0),
            'precision': m.get('metrics/precision(B)', 0),
            'recall':    m.get('metrics/recall(B)', 0),
            'box_loss':  m.get('train/box_loss', 0),
            'cls_loss':  m.get('train/cls_loss', 0),
        })

    # Log best weights as artifact
    best_pt = Path(PROJECT) / RUN_NAME / 'weights' / 'best.pt'
    if best_pt.exists():
        mlflow.log_artifact(str(best_pt), artifact_path='weights')
        print(f'\nbest.pt logged to MLflow: {best_pt}')

print('\nTraining complete!')

In [ ]:
# Cell 7 — View training results
from IPython.display import Image, display
import os

results_dir = f'{PROJECT}/{RUN_NAME}'

for img in ['results.png', 'confusion_matrix.png', 'val_batch0_pred.jpg']:
    path = os.path.join(results_dir, img)
    if os.path.exists(path):
        print(f'\n--- {img} ---')
        display(Image(path, width=900))

In [ ]:
# Cell 8 — Print final metrics summary
if hasattr(results, 'results_dict'):
    m = results.results_dict
    print('='*40)
    print('       Final Metrics Summary')
    print('='*40)
    print(f"  mAP50:     {m.get('metrics/mAP50(B)', 0):.4f}")
    print(f"  mAP50-95:  {m.get('metrics/mAP50-95(B)', 0):.4f}")
    print(f"  Precision: {m.get('metrics/precision(B)', 0):.4f}")
    print(f"  Recall:    {m.get('metrics/recall(B)', 0):.4f}")
    print(f"  Box loss:  {m.get('train/box_loss', 0):.4f}")
    print(f"  Cls loss:  {m.get('train/cls_loss', 0):.4f}")
    print('='*40)
    print(f'\nbest.pt saved at: {best_pt}')

In [ ]:
# Cell 9 — Copy best.pt to /kaggle/working for easy download
import shutil

dest = '/kaggle/working/best.pt'
shutil.copy(str(best_pt), dest)
print(f'Download best.pt from the Output tab on the right panel → {dest}')